In [1]:
import os
import json
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.decomposition import PCA

In [2]:
import os

os.makedirs("Models", exist_ok=True)

print("Models folder is ready.")

Models folder is ready.


In [3]:
############################################################
# CONFIG
############################################################

TRAIN_DIR = "/kaggle/input/datasets/vikas635233/dataset-for-train-test/EMG-EPN612 Dataset/trainingJSON"
TEST_DIR = "/kaggle/input/datasets/vikas635233/dataset-for-train-test/EMG-EPN612 Dataset/testingJSON"

TARGET_LEN = 1019
PCA_COMPONENTS = 6

BATCH_SIZE = 32
EPOCHS = 200
LR = 1e-3

############################################################
# LABEL MAP
############################################################

LABEL_MAP = {
    "noGesture": 0,
    "fist": 1,
    "waveIn": 2,
    "waveOut": 3,
    "open": 4,
    "pinch": 5
}

############################################################
# PAD / CROP
############################################################

def pad_or_crop(signal, target_len=TARGET_LEN):

    current_len = signal.shape[1]

    if current_len > target_len:

        signal = signal[:, :target_len]

    elif current_len < target_len:

        pad = target_len - current_len

        signal = np.pad(
            signal,
            ((0, 0), (0, pad)),
            mode='constant'
        )

    return signal

############################################################
# LOAD DATA
############################################################

def load_dataset(folder, allowed_users=None):



    X = []
    y = []

    for root, dirs, files in os.walk(folder):

 

        user_name = os.path.basename(root)

        if allowed_users is not None:
            if user_name not in allowed_users:
               
                continue

        for file in files:

            if not file.endswith(".json"):
                continue

            path = os.path.join(root, file)

        

            with open(path, "r") as f:
                data = json.load(f)

         

            if "trainingSamples" not in data:
               
                continue

            samples = data["trainingSamples"]

           
            for key in samples:

                sample = samples[key]

                gesture = sample["gestureName"]

                if gesture not in LABEL_MAP:
                    
                    continue

                emg = sample["emg"]

                signal = np.array([
                    emg["ch1"],
                    emg["ch2"],
                    emg["ch3"],
                    emg["ch4"],
                    emg["ch5"],
                    emg["ch6"],
                    emg["ch7"],
                    emg["ch8"]
                ], dtype=np.float32)

                signal = pad_or_crop(signal)

                X.append(signal)
                y.append(LABEL_MAP[gesture])


    return np.array(X, dtype=np.float32), np.array(y)

############################################################
# PCA
############################################################

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def apply_pca(X_train, X_test):

    scaler = StandardScaler()

    pca = PCA(
        n_components=PCA_COMPONENTS
    )

    temp = X_train.transpose(0,2,1)
    temp = temp.reshape(-1,8)

    temp = scaler.fit_transform(temp)

    pca.fit(temp)

    train_out = []

    for sample in X_train:

        sample = sample.T
        sample = scaler.transform(sample)
        sample = pca.transform(sample)
        sample = sample.T

        train_out.append(sample)

    test_out = []

    for sample in X_test:

        sample = sample.T
        sample = scaler.transform(sample)
        sample = pca.transform(sample)
        sample = sample.T

        test_out.append(sample)

    return (
        np.array(train_out, dtype=np.float32),
        np.array(test_out, dtype=np.float32)
    )

In [4]:
############################################################
# DATASET
############################################################

class EMGDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]



In [5]:
############################################################
# TERNARY HELPER FUNCTIONS
############################################################

import torch
import torch.nn as nn
import torch.nn.functional as F


class TernarySTE(torch.autograd.Function):

    @staticmethod
    def forward(ctx, weight):

        delta = 0.7 * weight.abs().mean()

        alpha = weight[weight.abs() > delta].abs().mean()

        if torch.isnan(alpha):
            alpha = torch.tensor(
                1.0,
                device=weight.device
            )

        output = torch.zeros_like(weight)

        output[weight > delta] = alpha

        output[weight < -delta] = -alpha

        return output

    @staticmethod
    def backward(ctx, grad_output):

        return grad_output


class TernaryConv1d(nn.Conv1d):

    def forward(self, x):

        ternary_weight = TernarySTE.apply(self.weight)

        return F.conv1d(
            x,
            ternary_weight,
            self.bias,
            self.stride,
            self.padding,
            self.dilation,
            self.groups
        )


class TernaryLinear(nn.Linear):

    def forward(self, x):

        ternary_weight = TernarySTE.apply(self.weight)

        return F.linear(
            x,
            ternary_weight,
            self.bias
        )

In [6]:
############################################################
# CNN MODEL
############################################################

class Ternary_PCA_CNN(nn.Module):

    def __init__(self):

        super().__init__()

        ############################################################
        # Depthwise + Pointwise Conv1
        ############################################################

        self.dw1 = TernaryConv1d(
            in_channels=6,
            out_channels=6,
            kernel_size=2,
            groups=6
        )

        self.pw1 = TernaryConv1d(
            in_channels=6,
            out_channels=32,
            kernel_size=1
        )

        self.pool = nn.MaxPool1d(
            kernel_size=2
        )

        ############################################################
        # Depthwise + Pointwise Conv2
        ############################################################

        self.dw2 = TernaryConv1d(
            in_channels=32,
            out_channels=32,
            kernel_size=2,
            groups=32
        )

        self.pw2 = TernaryConv1d(
            in_channels=32,
            out_channels=32,
            kernel_size=1
        )

        ############################################################
        # GAP
        ############################################################

        self.gap = nn.AdaptiveAvgPool1d(1)

        ############################################################
        # FC
        ############################################################

        self.fc1 = TernaryLinear(
            32,
            64
        )

        self.fc2 = TernaryLinear(
            64,
            6
        )

    def forward(self, x):

        x = F.relu(self.pw1(self.dw1(x)))

        x = self.pool(x)

        x = F.relu(self.pw2(self.dw2(x)))

        x = self.gap(x)

        x = x.squeeze(-1)

        x = F.relu(self.fc1(x))

        x = self.fc2(x)

        return x

In [7]:
model = Ternary_PCA_CNN()

print(model)

print("Parameters:",
      sum(p.numel() for p in model.parameters()))

Ternary_PCA_CNN(
  (dw1): TernaryConv1d(6, 6, kernel_size=(2,), stride=(1,), groups=6)
  (pw1): TernaryConv1d(6, 32, kernel_size=(1,), stride=(1,))
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dw2): TernaryConv1d(32, 32, kernel_size=(2,), stride=(1,), groups=32)
  (pw2): TernaryConv1d(32, 32, kernel_size=(1,), stride=(1,))
  (gap): AdaptiveAvgPool1d(output_size=1)
  (fc1): TernaryLinear(in_features=32, out_features=64, bias=True)
  (fc2): TernaryLinear(in_features=64, out_features=6, bias=True)
)
Parameters: 3896


In [8]:
############################################################
# LOAD DATA
############################################################
TEST_USERS = [
    "user1","user2","user3","user4","user5",
    "user6","user7","user8","user9","user10",
    "user11","user12","user13","user14","user15",
    "user16","user17","user18","user19","user20",
    "user21"
]
print("Loading data...")

print("TRAIN_DIR =", TRAIN_DIR)
print("TEST_DIR  =", TEST_DIR)

X_train, y_train = load_dataset(TRAIN_DIR)

X_test, y_test = load_dataset(
    TEST_DIR,
    TEST_USERS
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("Train distribution:")
print(np.unique(y_train, return_counts=True))

print("Test distribution:")
print(np.unique(y_test, return_counts=True))

############################################################
# PCA
############################################################

print("Applying PCA...")

X_train, X_test = apply_pca(
    X_train,
    X_test
)

np.save("X_train_pca.npy", X_train)
np.save("y_train.npy", y_train)

np.save("X_test_pca.npy", X_test)
np.save("y_test.npy", y_test)

print("After PCA")

print("Train :", X_train.shape)
print("Test  :", X_test.shape)

############################################################
# DATALOADER
############################################################

train_dataset = EMGDataset(
    X_train,
    y_train
)

test_dataset = EMGDataset(
    X_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

Loading data...
TRAIN_DIR = /kaggle/input/datasets/vikas635233/dataset-for-train-test/EMG-EPN612 Dataset/trainingJSON
TEST_DIR  = /kaggle/input/datasets/vikas635233/dataset-for-train-test/EMG-EPN612 Dataset/testingJSON
Train shape: (45900, 8, 1019)
Test shape : (3150, 8, 1019)
X_train: (45900, 8, 1019)
y_train: (45900,)
X_test : (3150, 8, 1019)
y_test : (3150,)
Train distribution:
(array([0, 1, 2, 3, 4, 5]), array([7650, 7650, 7650, 7650, 7650, 7650]))
Test distribution:
(array([0, 1, 2, 3, 4, 5]), array([525, 525, 525, 525, 525, 525]))
Applying PCA...
After PCA
Train : (45900, 6, 1019)
Test  : (3150, 6, 1019)


In [9]:
############################################################
# DEVICE
############################################################

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

############################################################
# MODEL
############################################################

model = Ternary_PCA_CNN().to(device)

total_params = sum(p.numel() for p in model.parameters())
print("Total Parameters:", total_params)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

############################################################
# TRAIN
############################################################

best_acc = 0

for epoch in range(EPOCHS):

    ########################################################
    # TRAIN
    ########################################################

    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for x, y in train_loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)

        loss = criterion(
            outputs,
            y
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        preds = outputs.argmax(1)

        train_total += y.size(0)

        train_correct += (
            preds == y
        ).sum().item()

    train_acc = (
        100.0 *
        train_correct /
        train_total
    )

    ########################################################
    # TEST
    ########################################################

    model.eval()

    test_correct = 0
    test_total = 0

    with torch.no_grad():

        for x, y in test_loader:

            x = x.to(device)
            y = y.to(device)

            outputs = model(x)

            preds = outputs.argmax(1)

            test_total += y.size(0)

            test_correct += (
                preds == y
            ).sum().item()

    test_acc = (
        100.0 *
        test_correct /
        test_total
    )

    ########################################################
    # PRINT RESULTS
    ########################################################

    print(
        f"Epoch {epoch+1:03d} | "
        f"Loss {train_loss:.4f} | "
        f"Train Acc {train_acc:.2f}% | "
        f"Test Acc {test_acc:.2f}%"
    )

    ########################################################
    # SAVE BEST MODEL
    ########################################################

    if test_acc > best_acc:

        best_acc = test_acc

        torch.save(
            model.state_dict(),
            "Models/best_model_ternary_depthwise.pth"
        )

        print(
            f"Saved model : {best_acc:.2f}%"
        )

print("\nTraining Complete")
print("Best Test Accuracy =", best_acc)

Device: cuda
Total Parameters: 3896
Epoch 001 | Loss 1899.1572 | Train Acc 43.41% | Test Acc 61.84%
Saved model : 61.84%
Epoch 002 | Loss 1183.9965 | Train Acc 68.55% | Test Acc 76.60%
Saved model : 76.60%
Epoch 003 | Loss 1000.6522 | Train Acc 74.55% | Test Acc 81.33%
Saved model : 81.33%
Epoch 004 | Loss 898.6696 | Train Acc 77.43% | Test Acc 82.89%
Saved model : 82.89%
Epoch 005 | Loss 823.0522 | Train Acc 79.23% | Test Acc 79.27%
Epoch 006 | Loss 788.8819 | Train Acc 80.17% | Test Acc 86.03%
Saved model : 86.03%
Epoch 007 | Loss 838.4247 | Train Acc 78.77% | Test Acc 83.97%
Epoch 008 | Loss 820.3786 | Train Acc 79.34% | Test Acc 85.14%
Epoch 009 | Loss 871.0401 | Train Acc 78.19% | Test Acc 82.83%
Epoch 010 | Loss 850.1621 | Train Acc 78.77% | Test Acc 85.94%
Epoch 011 | Loss 866.0474 | Train Acc 78.22% | Test Acc 84.60%
Epoch 012 | Loss 867.5565 | Train Acc 78.16% | Test Acc 77.52%
Epoch 013 | Loss 936.2858 | Train Acc 76.33% | Test Acc 81.68%
Epoch 014 | Loss 973.0693 | Train Acc

In [10]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

model.load_state_dict(
    torch.load("Models/best_model_ternary_depthwise.pth")
)

model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for x, y in test_loader:

        x = x.to(device)

        outputs = model(x)

        preds = outputs.argmax(1)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("Confusion Matrix")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report")
print(classification_report(y_true, y_pred))

Confusion Matrix
[[518   0   2   0   0   5]
 [  0 403   9   7  41  65]
 [  0  10 487   4   4  20]
 [  2   0  16 467  40   0]
 [  0  11  13  36 411  54]
 [  1   0   0   8  92 424]]

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       525
           1       0.95      0.77      0.85       525
           2       0.92      0.93      0.93       525
           3       0.89      0.89      0.89       525
           4       0.70      0.78      0.74       525
           5       0.75      0.81      0.78       525

    accuracy                           0.86      3150
   macro avg       0.87      0.86      0.86      3150
weighted avg       0.87      0.86      0.86      3150



In [11]:
print("Depthwise Conv1 :", model.dw1.weight.numel())
print("Pointwise Conv1 :", model.pw1.weight.numel())

print("Depthwise Conv2 :", model.dw2.weight.numel())
print("Pointwise Conv2 :", model.pw2.weight.numel())

print("FC1 :", model.fc1.weight.numel())
print("FC2 :", model.fc2.weight.numel())

print("\nTotal Parameters:",
      sum(p.numel() for p in model.parameters()))

Depthwise Conv1 : 12
Pointwise Conv1 : 192
Depthwise Conv2 : 64
Pointwise Conv2 : 1024
FC1 : 2048
FC2 : 384

Total Parameters: 3896
